# Inspect AutoDDG technical metadata

Set `CORPUS` to `inaturalist` or `pangaea`, and `INDEX` to `0`, `1`, `2`, ... through the sorted dataset list. The printed line is the species name or PANGAEA id for that index. The table is the notebook 03 ground truth beside each AutoDDG extraction.

In [ ]:
CORPUS = 'inaturalist'  # 'inaturalist' or 'pangaea'
INDEX = 3

import json
from pathlib import Path

import pandas as pd

REPO = Path('/home/com3dian/Github/process_data_metadata_agent')
TECH = REPO / 'outputs' / 'technical_from_descriptions'
INAT_DATA = REPO / 'outputs' / 'inaturalist' / 'inaturalist_100_species_1000_obs'
EXTENTS = REPO / 'outputs' / 'pangaea' / 'sample_20260629_163144_extents.json'
FIELDS = ['min_lat', 'max_lat', 'min_lon', 'max_lon', 'min_time', 'max_time']
MODEL_NAMES = {
    'default-text-large': 'Llama3 70B',
    'Qwen_Qwen3.6-35B-A3B-FP8': 'Qwen 3.6 35B',
    'Sehyo_Qwen3.5-122B-A10B-NVFP4': 'Qwen 3.5 122B',
    'mistralai_Mistral-Small-3.2-24B-Instruct-2506': 'Mistral Small 3.2 24B',
    'mistralai_Devstral-Small-2-24B-Instruct-2512': 'Mistral Devstral 2 24B',
}

def predicted_fields(path):
    empty = {field: None for field in FIELDS}
    if not path.exists():
        return empty
    payload = json.loads(path.read_text())
    spatial = payload.get('spatialCoverage') or {}
    temporal = payload.get('temporalCoverage') or {}
    if not isinstance(spatial, dict):
        spatial = {}
    if not isinstance(temporal, dict):
        temporal = {}
    return {
        'min_lat': spatial.get('min_lat'),
        'max_lat': spatial.get('max_lat'),
        'min_lon': spatial.get('min_lon'),
        'max_lon': spatial.get('max_lon'),
        'min_time': temporal.get('from'),
        'max_time': temporal.get('to'),
    }

if CORPUS == 'pangaea':
    dataset_ids = sorted(json.loads(EXTENTS.read_text()), key=int)
    if not 0 <= INDEX < len(dataset_ids):
        raise IndexError(f'INDEX {INDEX} is outside 0..{len(dataset_ids) - 1} for PANGAEA')
    dataset_id = dataset_ids[INDEX]
    extents = json.loads(EXTENTS.read_text())
    spatial = extents[dataset_id]['spat_extent'] or {}
    temporal = extents[dataset_id]['temp_extent'] or [None, None]
    truth = {
        'min_lat': spatial.get('southBoundLatitude'),
        'max_lat': spatial.get('northBoundLatitude'),
        'min_lon': spatial.get('westBoundLongitude'),
        'max_lon': spatial.get('eastBoundLongitude'),
        'min_time': temporal[0],
        'max_time': temporal[1],
    }
    runs = sorted((TECH / 'pangaea').glob('autoddg_*'))
    filename = f'{dataset_id}_pangaea.json'
    prefix = 'autoddg_pangaea_'
    label = f'INDEX {INDEX} of {len(dataset_ids)}  pangaea {dataset_id}'
elif CORPUS == 'inaturalist':
    dataset_ids = sorted(path.stem for path in INAT_DATA.glob('*.csv'))
    if not 0 <= INDEX < len(dataset_ids):
        raise IndexError(f'INDEX {INDEX} is outside 0..{len(dataset_ids) - 1} for iNaturalist')
    dataset_id = dataset_ids[INDEX]
    csv_path = INAT_DATA / f'{dataset_id}.csv'
    observations = pd.read_csv(csv_path)
    event_time = pd.to_datetime(observations['eventDate'], errors='coerce')
    truth = {
        'min_lat': observations['decimalLatitude'].min(),
        'max_lat': observations['decimalLatitude'].max(),
        'min_lon': observations['decimalLongitude'].min(),
        'max_lon': observations['decimalLongitude'].max(),
        'min_time': event_time.min(),
        'max_time': event_time.max(),
    }
    runs = sorted((TECH / 'inaturalist').glob('autoddg_*'))
    filename = f'{dataset_id}_inaturalist.json'
    prefix = 'autoddg_inat_'
    label = f'INDEX {INDEX} of {len(dataset_ids)}  inaturalist {dataset_id}'
else:
    raise ValueError("CORPUS must be 'inaturalist' or 'pangaea'")

columns = {'ground truth': [truth[field] for field in FIELDS]}
for run in runs:
    model = MODEL_NAMES.get(run.name.removeprefix(prefix), run.name)
    fields = predicted_fields(run / filename)
    columns[model] = [fields[field] for field in FIELDS]

comparison = pd.DataFrame(columns, index=FIELDS)
print(label)
comparison


INDEX 2 of 100  inaturalist acrothamnus_colensoi


,ground truth,Qwen 3.6 35B,Qwen 3.5 122B,Llama3 70B,Mistral Devstral 2 24B,Mistral Small 3.2 24B
min_lat,-45.848455,None,None,-44.73,None,None
max_lat,-39.145751,None,None,-41.10,None,None
min_lon,167.669761,None,None,0.00,None,None
max_lon,176.213639,None,None,173.49,None,None
min_time,2004-01-24 14:09:00,None,2019-01-01T00:00:00,NaN,None,2019-01-01T00:00:00
max_time,2026-05-23 12:21:00,None,2026-12-31T23:59:59,NaN,None,2026-12-31T23:59:59


In [6]:
spatial_columns = [
    'countryCode', 'locality', 'stateProvince',
    'decimalLatitude', 'decimalLongitude',
    'coordinateUncertaintyInMeters', 'coordinatePrecision',
    'elevation', 'elevationAccuracy', 'depth', 'depthAccuracy',
]
temporal_columns = ['eventDate', 'day', 'month', 'year']
selected = observations[spatial_columns + temporal_columns].copy()
selected['eventDate'] = pd.to_datetime(selected['eventDate'], errors='coerce')
print(dataset_id)
pd.DataFrame({'min': selected.min(), 'max': selected.max()})

acrothamnus_colensoi


,min,max
countryCode,NZ,NZ
locality,NaN,NaN
stateProvince,Canterbury,West Coast
decimalLatitude,-45.848455,-39.145751
decimalLongitude,167.669761,176.213639
coordinateUncertaintyInMeters,1.0,57735.0
coordinatePrecision,NaN,NaN
elevation,NaN,NaN
elevationAccuracy,NaN,NaN
depth,NaN,NaN
